# FSOT Biohub v50 — CPU competitive pipeline

**Kaggle is CPU-only.** Train/tune the U-Net on GPU locally; this notebook runs inference on CPU only.

Pipeline: **U-Net FT detect** → **FSOT linking** → **ML division refine (f47)** → **ILP** → **division-gap patch** → `submission.csv`

Local train-proxy (`44b6_0113de3b`): **0.979** edge score. Full 4-dataset run: **~312,387 rows**.

**Inputs:** competition test + `cellmot-ft-detector-biohub` + `cellmot-baseline-artifacts` + `fsot-v50-competitive-bundle`

In [ ]:
import os
import sys
import shutil
import glob
from pathlib import Path

# Silence Kaggle Jupyter debugger noise (no effect on inference time)
os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")
os.environ.setdefault("PYTHONWARNINGS", "ignore::nbformat.MissingIDFieldWarning")

WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
os.environ.setdefault("KAGGLE_CPU_ONLY", "1")
os.environ.setdefault("CELLMOT_DEVICE", "cpu")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("TORCH_NUM_THREADS", "4")
os.environ.setdefault("CELLMOT_DET_TTA", "0")
os.environ.setdefault("BIOHUB_ENGINE", "fsot_unet")
os.environ.setdefault("FSOT_VISION_CALIBRATE", "1")
os.environ.setdefault("FSOT_LIVING_EMERGENCE", "1")
os.environ.setdefault("FSOT_LIVING_ADAPTIVE", "1")
os.environ.setdefault("FSOT_DET_CONF_RANK", "1")
os.environ.setdefault("FSOT_LIVING_PROXY_ACCURACY", "0.90")
os.environ.setdefault("FSOT_LINK_MODE", "fsot")
os.environ.setdefault("CELLMOT_USE_FT", "1")
os.environ.setdefault("CELLMOT_DET_THRESHOLD", "0.48")
os.environ.setdefault("CELLMOT_EDGE_THRESHOLD", "0.25")
os.environ.setdefault("CELLMOT_NMS_UM", "6.0")
os.environ.setdefault("CELLMOT_USE_ILP", "1")
os.environ.setdefault("CELLMOT_ILP_MAX_EDGES", "80000")
os.environ.setdefault("FSOT_GAP_LINK", "1")
os.environ.setdefault("FSOT_DIVISION_ML_REFINE", "1")
os.environ.setdefault("FSOT_ML_REFINE_FRAMES", "47")
os.environ.setdefault("FSOT_ML_REFINE_REPLACE_FRAMES", "47")
os.environ.setdefault("FSOT_ML_REFINE_EDGE_THRESHOLD", "0.05")
os.environ.setdefault("FSOT_ML_PRESERVE_FSOT_FRAMES", "29")
os.environ.setdefault("FSOT_ML_DIVISION_GAP_PATCH", "1")
os.environ.setdefault("FSOT_MITOSIS_VELOCITY", "1")
os.environ.setdefault("KAGGLE_SUBMISSION_FAST_VALIDATE", "1")

def _first_existing(paths):
    for p in paths:
        if p and Path(p).exists():
            return Path(p)
    return None

bundle_dir = _first_existing([
    "/kaggle/input/datasets/damianpalumbo/fsot-v50-competitive-bundle",
    "/kaggle/input/fsot-v50-competitive-bundle",
])
if bundle_dir is None:
    hits = glob.glob("/kaggle/input/**/kaggle_main_runner.py", recursive=True)
    bundle_dir = Path(hits[0]).parent if hits else None
if bundle_dir is None:
    raise FileNotFoundError("Attach damianpalumbo/fsot-v50-competitive-bundle")

for name in [
    "kaggle_main_runner.py", "biohub_unet_engine.py", "biohub_competitive.py",
    "fsot_division_ml_refine.py", "fsot_cellular_bridge.py", "fsot_core.py",
    "fsot_living_emergence.py", "fsot_vision_calibrate.py", "fsot_original_competition.py",
    "submission_io.py", "validate_kaggle_submission.py", "csv_to_geffs.py",
    "kaggle_wheel_bootstrap.py",
]:
    src = bundle_dir / name
    if src.exists():
        shutil.copy2(src, WORK / name)
print(f"[BUNDLE] {bundle_dir}")

weights = _first_existing([
    "/kaggle/input/datasets/aashishnegi23/cellmot-ft-detector-biohub/edge_predictor_best.pth",
    "/kaggle/input/cellmot-ft-detector-biohub/edge_predictor_best.pth",
])
if weights is None:
    hits = glob.glob("/kaggle/input/**/edge_predictor_best.pth", recursive=True)
    weights = Path(hits[0]) if hits else None
if weights is None:
    raise FileNotFoundError("cellmot-ft-detector-biohub weights not found")
os.environ["CELLMOT_UNET_WEIGHTS"] = str(weights)
print(f"[UNET] {weights}")

sys.path.insert(0, str(WORK))
from kaggle_wheel_bootstrap import extract_cellmot_bundle, install_cellmot_wheels

wheel_dir = install_cellmot_wheels()
if wheel_dir is None:
    raise RuntimeError("cellmot-baseline-artifacts wheels required")
if not extract_cellmot_bundle(WORK):
    raise RuntimeError("cellmot_bundle not found")

import predict_unet_transformer  # noqa: F401 — must resolve before runner starts
print("[SETUP] predict_unet_transformer OK")

In [ ]:
import runpy
import sys

sys.path.insert(0, "/kaggle/working")
runpy.run_path("/kaggle/working/kaggle_main_runner.py", run_name="__main__")

import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
print(sub.groupby(["dataset", "row_type"]).size())
print(f"submission rows: {len(sub)}")